# Support Vector Machine: Predicting Hard-Drive Failure
**IST 707 Final Project**

This notebook trains a support vector machine (SVM) to predict whether a hard drive will fail in the next 30 days, using its SMART health readings from the last 30 days. It uses the same data, inputs and time-based folds as the tree models, so they can be compared directly. It:

1. loads the training and calibration splits (the 2025 test split stays closed until final evaluation)
2. builds the model inputs, following the EDA's decisions, then log-transforms and scales them, which an SVM needs and a tree does not
3. trains a first SVM and checks it on the calibration split
4. tunes the SVM's settings with a grid search over time-based folds
5. saves the best model

**Why an approximate kernel.** A standard kernel SVM (scikit-learn's `SVC`) gets slower roughly with the square of the number of rows, so at 170,000 training windows a single fit could take hours, and tuning needs hundreds. Instead, `Nystroem` builds an approximation of the RBF kernel's features from a sample of the data, and a fast linear SVM (`LinearSVC`) is trained on them. The result is still a curved, nonlinear SVM, and each fit takes seconds to minutes. A plain linear SVM was not used because it draws the same kind of straight-line boundary as the logistic regression baseline and would add little to the comparison.

## 1. Libraries
Imports everything the notebook uses:
- **pandas / numpy:** loading and handling the data
- **scikit-learn:**
  - the pipeline pieces (`Pipeline`, `ColumnTransformer`, `FunctionTransformer`, `OneHotEncoder`, `SimpleImputer`, `StandardScaler`)
  - the kernel approximation (`Nystroem`) and the model (`LinearSVC`)
  - the tuning tool (`GridSearchCV`) and the scoring metrics
- **time / joblib:** timing the search and saving the final model to a file

In [10]:
import numpy as np
import pandas as pd
import time
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.svm import LinearSVC
from sklearn.kernel_approximation import Nystroem
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
)

## 2. Loading the Data
Loads the two splits used for training and checking. **The test split is not loaded.**

- **`modeling_train.parquet`:** Jan 2023 – Oct 2024. 169,827 drive-windows, 8,087 failures. Healthy windows were down-sampled to 20 per failure, so failures are 1 in 21.
- **`modeling_calib.parquet`:** Nov – Dec 2024. 297,009 drive-windows, 328 failures, at the real failure rate of about 1 in 905.

For each split the cell creates:
- **`X_..._raw`:** every column except the answer
- **`y_...`:** the answer, `label` (1 = the drive fails in the next 30 days)
- **`meta_...`:** serial number, window number and model. These are **not** model inputs; they're used to build the time-based folds.

In [11]:
train = pd.read_parquet("../../data/modeling/modeling_train.parquet")
calib = pd.read_parquet("../../data/modeling/modeling_calib.parquet")

X_train_raw, y_train = train.drop(columns="label"), train["label"].astype(int)
X_calib_raw, y_calib = calib.drop(columns="label"), calib["label"].astype(int)

meta_train = train[["serial_number", "window", "model"]].reset_index(drop=True)
meta_calib = calib[["serial_number", "window", "model"]].reset_index(drop=True)

print("Training shape:", X_train_raw.shape)
print("Calibration shape:", X_calib_raw.shape)

display(X_train_raw.head())
display(X_calib_raw.head())

display(y_train.head())
display(y_calib.head())

Training shape: (169827, 93)
Calibration shape: (297009, 93)


,serial_number,window,smart_10_normalized_first,smart_10_normalized_last,smart_10_raw_first,smart_10_raw_last,smart_12_normalized_first,smart_12_normalized_last,smart_12_raw_first,smart_12_raw_last,...,smart_3_raw_delta,smart_4_normalized_delta,smart_4_raw_delta,smart_5_normalized_delta,smart_5_raw_delta,smart_7_normalized_delta,smart_7_raw_delta,smart_9_normalized_delta,smart_9_raw_delta,drive_days_to_date
0,1050A093F97G,0,100.0,100.0,0.0,0.0,100.0,100.0,2.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,698.0,30
1,1050A0ANF97G,0,100.0,100.0,0.0,0.0,100.0,100.0,5.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,697.0,30
2,1080A06JF97G,0,100.0,100.0,0.0,0.0,100.0,100.0,6.0,6.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,697.0,30
3,1080A075F97G,0,100.0,100.0,0.0,0.0,100.0,100.0,7.0,7.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,681.0,30
4,1080A09LF97G,0,100.0,100.0,0.0,0.0,100.0,100.0,2.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,697.0,30


,serial_number,window,smart_10_normalized_first,smart_10_normalized_last,smart_10_raw_first,smart_10_raw_last,smart_12_normalized_first,smart_12_normalized_last,smart_12_raw_first,smart_12_raw_last,...,smart_3_raw_delta,smart_4_normalized_delta,smart_4_raw_delta,smart_5_normalized_delta,smart_5_raw_delta,smart_7_normalized_delta,smart_7_raw_delta,smart_9_normalized_delta,smart_9_raw_delta,drive_days_to_date
0,14A0A001F9RG,22,100.0,100.0,0.0,0.0,100.0,100.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,119.0,6
1,14B0A005F9RG,22,100.0,100.0,0.0,0.0,100.0,100.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,139.0,7
2,1PG6SGEV,22,100.0,100.0,0.0,0.0,100.0,100.0,8.0,8.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,78.0,5
3,1PG700ZV,22,100.0,100.0,0.0,0.0,100.0,100.0,10.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,94.0,5
4,1PG73SEV,22,100.0,100.0,0.0,0.0,100.0,100.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,171.0,8


0    0
1    0
2    0
3    0
4    0
Name: label, dtype: int64

0    0
1    0
2    0
3    0
4    0
Name: label, dtype: int64

## 3. Building the Model Inputs
Defines how the raw table is turned into model inputs. This only sets rules; nothing is calculated from the data here.

- **`KEEP_RAW`:** the SMART attributes kept, as raw values only. The EDA showed the vendor-normalized values lose detail and partly encode the brand. S10 is left out because it is always 0.
- **`vendor()`:** reads the manufacturer (Seagate, Toshiba, WDC, HGST) from the model name.
- **`build_features()`:**
  - keeps the first, last and change values for each kept attribute
  - keeps S9 (power-on hours) as drive age
  - turns the unknown capacity of -1 into a blank (23 training rows)
  - keeps days observed and the vendor
- **`signed_log()`:** computes `sign(x) · log(1 + |x|)`. Raw SMART counters span many orders of magnitude (some Seagate counters reach the billions), and the change columns can be negative. The log squeezes the huge values down while keeping the sign. This is the same transform the logistic regression baseline uses.
- **`NUMERIC` / `CATEGORICAL`:** the lists of number columns and the one text column (vendor).
- **`features` / `vendor_onehot`:** wrap these rules so they can sit inside the pipeline. The one-hot encoder turns vendor into yes/no columns, using a fixed vendor list so train and calibration always get the same columns.

In [12]:
KEEP_RAW = [1, 3, 4, 5, 7, 12, 192, 193, 194, 197, 198, 199]   # raw only; S9 -> age, S10 always 0
VENDORS = ["Seagate", "Toshiba", "WDC", "HGST", "Other"]


def vendor(model):
    m = str(model)
    if m.startswith("ST"): return "Seagate"
    if m.startswith("TOSHIBA"): return "Toshiba"
    if m.startswith(("WDC", "WUH")): return "WDC"
    if m.startswith(("HGST", "Hitachi")): return "HGST"
    return "Other"

## This Function Keeps raw SMART values only. Drops useless columns. Calculates drive age through power-on hours and cleans the storage column.
def build_features(df):
    X = pd.DataFrame(index=df.index)
    for i in KEEP_RAW:
        for s in ("first", "last", "delta"):
            X[f"s{i}_{s}"] = df[f"smart_{i}_raw_{s}"].astype("float64")
    X["s9_last"] = df["smart_9_raw_last"].astype("float64")          
    X["capacity_tb"] = (df["capacity_bytes"] / 1e12).round(0).where(df["capacity_bytes"] > 0)   # -1 = unknown -> NaN
    X["days_observed"] = df["days_observed"].astype("float64")
    X["vendor"] = df["model"].map(vendor)
    return X

def signed_log(X):
    return np.sign(X) * np.log1p(np.abs(X))

NUMERIC = [f"s{i}_{s}" for i in KEEP_RAW for s in ("first", "last", "delta")] + \
          ["s9_last", "capacity_tb", "days_observed"]

CATEGORICAL = ["vendor"]

features = FunctionTransformer(build_features)
vendor_onehot = OneHotEncoder(categories=[VENDORS], handle_unknown="ignore", sparse_output=False)

## 4. Preprocessing Pipeline
Chains the preprocessing steps into one object, `svm_preprocess`:

1. **features:** applies `build_features` to the raw table
2. **columns:**
   - **numbers**, in three steps:
     1. fills blanks with the **training** median and adds a "was missing" column for each column that had blanks. The blanks come from specific drive models, such as S197 on two Toshiba models, so the fact that a value is missing is itself useful.
     2. applies `signed_log`
     3. scales every column to mean 0 and standard deviation 1 (`StandardScaler`)
   - **vendor:** one-hot encoded

**Why scaling matters here and didn't for the trees.** An SVM works with distances between drive-windows. Without scaling, a column measured in billions would decide every distance by itself and the other columns would barely count. A tree only ever compares one column against a cutoff, so it doesn't care about scale.

Because this sits inside a pipeline, the medians, means and standard deviations are learned from training data only. They're then re-used on the calibration split, so no information leaks from calibration into training.

In [13]:
svm_preprocess = Pipeline([
    ("features", features),
    ("columns", ColumnTransformer([
        ("numeric", Pipeline([
            ("impute", SimpleImputer(strategy="median", add_indicator=True)), 
            ("log", FunctionTransformer(signed_log)), 
            ("scale", StandardScaler())
            ]),
            NUMERIC
        ),
        ("vendor", vendor_onehot, CATEGORICAL),
    ], verbose_feature_names_out=False)),
]).set_output(transform="pandas")

## 5. First SVM (Before Tuning)
Builds the full model in three steps, with reasonable starting settings:

1. **`preprocessing`:** `svm_preprocess`, from section 4
2. **`kernel`:** `Nystroem` approximates the RBF kernel. The RBF kernel scores two windows as similar when their scaled readings are close, which lets the SVM draw a curved boundary between healthy and failing drives.
   - **`kernel="rbf"`:** the kernel being approximated
   - **`n_components=500`:** how many sample points the approximation is built from. More is closer to the real kernel but slower.
   - **`random_state=42`:** fixes which points are sampled, so results repeat
3. **`classifier`:** `LinearSVC` finds the widest margin between the classes in the approximated kernel space
   - **`C=1.0`:** how heavily windows on the wrong side of the margin are penalized. Higher values fit the training data more tightly; lower values allow a wider, simpler margin.
   - **`class_weight="balanced"`:** gives failures more weight
   - **`max_iter=5000`:** more solver steps than the default, so the fit can finish
   - **`random_state=42`:** makes results repeatable

In [14]:
svm_model = Pipeline([
    ("preprocessing", svm_preprocess),
    ("kernel", Nystroem(kernel = "rbf", n_components = 500, random_state = 42)),
    ("classifier", LinearSVC(C = 1.0, class_weight = 'balanced', max_iter = 5000, random_state = 42))
])

### 5.1 Training the First Model
Fits the whole pipeline on the training split: the preprocessing learns its medians and scaling, the kernel approximation picks its sample points, then the SVM finds its margin.

**Result:** the fit took **0.6 minutes**. The search runs 96 fits, so this is what sets its length.

In [15]:

start = time.time()
svm_model.fit(X_train_raw, y_train)
print(f"\nFit took {(time.time() - start) / 60:.1f} minutes")

/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b



Fit took 0.7 minutes


### 5.2 Checking the First Model on the Calibration Split
An SVM doesn't produce probabilities. Instead `decision_function` gives each window a **margin score**: how far it sits from the boundary, and on which side. 0 is the boundary itself, positive means the failure side, and -1 and +1 are the edges of the margin.

The cell scores every calibration window, then turns the scores into yes/no predictions at margin thresholds from -1.5 to 1.5. At each threshold it records:

- **PR-AUC (AP):** how well the model *ranks* failing drives above healthy ones, over all thresholds. This is our main metric, and random guessing scores 0.0011. It only needs a ranking, so margin scores work as well as probabilities.
- **ROC-AUC:** another ranking score (0.5 = coin flip, 1.0 = perfect). It looks good easily when failures are rare, so it's secondary.
- **Precision:** of the drives flagged, the share that actually failed
- **Recall:** of the drives that failed, the share that were flagged
- **F1:** one number balancing precision and recall

PR-AUC and ROC-AUC don't depend on the threshold, which is why they're the same in every row. Only precision, recall and F1 change.

**No Brier score.** The Brier score measures how close predicted *probabilities* are to what happened, and margin scores aren't probabilities. The SVM gets probabilities at the calibration stage, the same way every model's probabilities are corrected there.

**Result:**
- **Ranking:** PR-AUC is **0.0336**, against 0.0073 for the SMART 5 rule, 0.0011 for random, 0.082 for logistic regression, 0.0701 for the untuned decision tree and 0.0995 for the untuned random forest. ROC-AUC is 0.879. So the untuned SVM ranks failing drives 5 times better than the SMART 5 rule, but well below every other trained model. The tuning in section 6 shows why: the starting kernel setting (`gamma` of about 0.02) lets the boundary bend too sharply, and every one of the best tuned settings uses a much smaller `gamma`.
- **Thresholds:** at the SVM's own boundary (0), precision is 1.1% at 75.9% recall. The best F1 (0.074) is at 1.5, with precision 6.1% and recall 9.5%. Like every model here, the SVM was trained with failures 43× more common than in reality, so its boundary sits too far toward flagging drives, and a threshold has to be chosen deliberately.

In [16]:
results=[]
y_score = svm_model.decision_function(X_calib_raw)

# Default decision threshold
for x in [-1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5]:
    y_pred = (y_score >= x).astype(int)

    results.append({
        "Threshold": x,
        "Model": "SVM",
        "PR-AUC (AP)": average_precision_score(y_calib, y_score),
        "ROC-AUC": roc_auc_score(y_calib, y_score),
        "Precision": precision_score(
            y_calib, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_calib, y_pred, zero_division=0
        ),
        "F1": f1_score(y_calib, y_pred, zero_division=0),
    })

results_df = pd.DataFrame(results)
display(results_df.round(4))

/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


,Threshold,Model,PR-AUC (AP),ROC-AUC,Precision,Recall,F1
0,-1.5,SVM,0.0336,0.8791,0.0011,1.0000,0.0022
1,-1.0,SVM,0.0336,0.8791,0.0011,1.0000,0.0022
2,-0.5,SVM,0.0336,0.8791,0.0025,0.8720,0.0050
3,0.0,SVM,0.0336,0.8791,0.0110,0.7591,0.0217
4,0.5,SVM,0.0336,0.8791,0.0201,0.6616,0.0390
5,1.0,SVM,0.0336,0.8791,0.0415,0.2957,0.0728
6,1.5,SVM,0.0336,0.8791,0.0614,0.0945,0.0744


## 6. Tuning the SVM
### 6.1 Search Setup and Time-Based Folds
Sets up the tuning:

- **`svm_model`:** the search reuses the model from section 5. It works on a fresh copy, so the first model is left untouched, and each combination overwrites the settings being searched. `class_weight="balanced"`, `max_iter` and the random seeds carry over to every combination.
- **`param_grid`:** the settings to try:
  - **`C`:** how heavily windows on the wrong side of the margin are penalized
  - **`gamma`:** how far one training window's influence reaches in the RBF kernel. Small values give a smooth, gently curved boundary; large values let it bend around individual windows, which risks memorizing. The default is 1 divided by the number of columns, about 0.02 here.
  - **`n_components`:** how detailed the kernel approximation is

  The grid shown is the second of two. The first tried 24 combinations: `C` of 0.01, 0.1, 1 and 10, `gamma` of 0.001, 0.01 and 0.1, and 300 or 1000 components. Its best result sat on the edge of the grid for all three settings (the smallest `gamma`, the largest `C`, the most components), so the true best could have been outside it. The second grid moves past those edges: `C` of 1, 10 and 100, `gamma` of 0.0001, 0.0003 and 0.001, and 1000 components only, which every top-3 result in the first search used. That's 9 combinations. An SVM has far fewer settings worth tuning than a tree, so every combination is tried.
- **`time_folds()`:** builds 4 cross-validation folds **by time**. Each fold trains on earlier months and validates on later months, with a one-window gap in between, because each label looks one window ahead. This mirrors real use: predicting the future from the past.

The printout shows the 4 folds growing: training uses more months each time, and each validation block covers the next few months.

In [17]:
param_grid = {
    "classifier__C": [1, 10, 100],
    "kernel__gamma": [0.0001, 0.0003, 0.001],
    "kernel__n_components": [1000]
}

## Creating a cross validation set.
def time_folds(meta, n_splits=4, gap=1):
    """Expanding-window folds by calendar window. Skips `gap` windows between train and validation,
    because a window's label looks one window ahead."""
    w = meta["window"].to_numpy()
    windows = np.sort(np.unique(w))
    blocks = np.array_split(windows, n_splits + 1)
    folds = []
    for k in range(1, n_splits + 1):
        valid_w = blocks[k]
        train_w = windows[windows < valid_w.min() - gap]
        folds.append((np.flatnonzero(np.isin(w, train_w)), np.flatnonzero(np.isin(w, valid_w))))
    return folds

cv_folds = time_folds(meta_train)
for k, (tr_idx, va_idx) in enumerate(cv_folds):
    print(f"fold {k}: train {len(tr_idx):,} rows -> validate {len(va_idx):,} rows")


fold 0: train 28,508 rows -> validate 37,027 rows
fold 1: train 64,596 rows -> validate 31,106 rows
fold 2: train 95,671 rows -> validate 32,768 rows
fold 3: train 128,057 rows -> validate 33,501 rows


### 6.2 Grid Search, Results and Saving the Best Model
- **Grid search:** tries **all 9 combinations**. Each is trained once per fold, giving **36 fits**, and scored by PR-AUC (`average_precision`), which the search computes from the margin scores. The tree models used a randomized search because their grids were too large to try in full; with only 24 combinations there is no need to sample. `n_jobs=4` runs 4 fits at once: the kernel approximation of a fold's training rows can exceed 1 GB, so running every fit in parallel could run out of memory.
- **Refit:** the best combination is retrained on all of the training data (`refit=True`).
- **Results table:**
  - `mean_test_score`: the average PR-AUC over the 4 folds
  - `std_test_score`: how much the 4 fold scores vary
  - `rank_test_score`: 1 = best

  The table is saved to `svm_search_results_ranked.csv`.
- **Save:** the best model is saved to `svm_best.joblib`, so it can be reloaded without retraining.

**Result:**
- **Time:** the search took **31.7 minutes**. The first, 24-combination search took 12.6; this one is slower because large `C` values take the solver much longer, up to 7 minutes per fit.
- **Best settings:** **`C=100`, `gamma=0.0003`, 1000 components**, with a mean time-CV PR-AUC of **0.551** (std 0.029). That is level with the tuned decision tree (0.548) and below the tuned random forest (0.573).
- **The search has found the plateau.** The first search's best (`C=10`, `gamma=0.001`, 0.550) is now second, and the top 5 settings are within 0.004 of each other, far less than one std apart. `gamma=0.0003` sits between two worse values, and raising `C` from 10 to 100 adds almost nothing, so widening the grid further would not help.
- **A faster equivalent:** `C=10`, `gamma=0.001` scores 0.550 and fits in about a minute, against three minutes for the best setting.
- **What the best settings mean:** every top result uses a very small `gamma`, so each training window's influence reaches far and the boundary is smooth and only gently curved. The SVM does best when it stays close to a straight-line boundary.

These scores come from training folds, where failures are 1 in 21, so they're much higher than PR-AUC on the calibration split. Use them only to compare settings with each other.

In [18]:
search = GridSearchCV(
    estimator=svm_model,
    param_grid=param_grid,
    scoring={
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "pr_auc": "average_precision",
    },
    cv=cv_folds,
    n_jobs=4,
    refit="pr_auc",
    verbose=2,
    return_train_score=False,
)

start = time.time()
search.fit(X_train_raw, y_train)
print(f"\nSearch took {(time.time() - start) / 60:.1f} minutes")

cv_results = pd.DataFrame(search.cv_results_)

# All four metrics, averaged across the CV folds (column names follow the scoring keys)
results = pd.DataFrame({
    "PR_AUC":         cv_results["mean_test_pr_auc"],
    "PR_AUC_std":     cv_results["std_test_pr_auc"],
    "Precision":      cv_results["mean_test_precision"],
    "Precision_std":  cv_results["std_test_precision"],
    "Recall":         cv_results["mean_test_recall"],
    "Recall_std":     cv_results["std_test_recall"],
    "F1":             cv_results["mean_test_f1"],
    "F1_std":         cv_results["std_test_f1"],
    "mean_fit_time":  cv_results["mean_fit_time"],
})

for col in cv_results.columns:
    if col.startswith("param_"):
        results[col] = cv_results[col]

# Rank by PR-AUC (the same ranking the search used to pick the best model)
results["PR_AUC_rank"] = cv_results["rank_test_pr_auc"]
results = results.sort_values("PR_AUC_rank").reset_index(drop=True)

print("Best PR-AUC (time-CV on train):", round(search.best_score_, 4))
print("Best parameters:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")

print("\nTop 10 models ranked by PR-AUC:")
display(results.head(10))

results.to_csv("svm_search_results_ranked.csv", index=False)

# Save the best model (chosen by PR-AUC, refit on all of train)
best_svm = search.best_estimator_
joblib.dump(best_svm, "svm_best.joblib")


Fitting 4 folds for each of 9 candidates, totalling 36 fits


/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-

[CV] END classifier__C=1, kernel__gamma=0.0001, kernel__n_components=1000; total time=   3.7s


/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-

[CV] END classifier__C=1, kernel__gamma=0.0003, kernel__n_components=1000; total time=   4.3s


/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extma

... (65 outputs left out)

/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.11/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b



Search took 31.7 minutes
Best PR-AUC (time-CV on train): 0.551
Best parameters:
  classifier__C: 100
  kernel__gamma: 0.0003
  kernel__n_components: 1000

Top 10 models ranked by PR-AUC:


,PR_AUC,PR_AUC_std,Precision,Precision_std,Recall,Recall_std,F1,F1_std,mean_fit_time,param_classifier__C,param_kernel__gamma,param_kernel__n_components,PR_AUC_rank
0,0.550983,0.028902,0.341372,0.037236,0.683264,0.034661,0.454645,0.039021,184.323093,100,0.0003,1000,1
1,0.550153,0.033656,0.340180,0.035510,0.686082,0.034930,0.454275,0.037661,68.384189,10,0.0010,1000,2
2,0.549238,0.024375,0.346040,0.035081,0.674738,0.035060,0.456872,0.036450,95.419526,100,0.0001,1000,3
3,0.548217,0.025133,0.346166,0.033875,0.675163,0.035783,0.457143,0.035694,40.897788,10,0.0003,1000,4
4,0.547506,0.035745,0.326249,0.035117,0.689054,0.034793,0.442270,0.038183,424.246473,100,0.0010,1000,5
5,0.545404,0.028331,0.346498,0.033661,0.675665,0.038081,0.457554,0.035860,22.662300,1,0.0010,1000,6
6,0.539578,0.021083,0.355414,0.025180,0.657779,0.032766,0.461174,0.027062,30.836719,10,0.0001,1000,7
7,0.537020,0.020251,0.359508,0.022666,0.656145,0.034276,0.464214,0.025009,17.418099,1,0.0003,1000,8
8,0.524873,0.017227,0.371936,0.021844,0.641403,0.034940,0.470612,0.024811,12.114050,1,0.0001,1000,9


['svm_best.joblib']